# ETF 滚动 OLS、AR(1) 及信号特征

读取data/etf_daily_full_coverage_483.parquet的完整历史，计算日频残差动力学特征。方法见README.md及docs/ETF均值回归增量预测研究报告.md。

沿用现有close_hfq及质量标记。s为连续得分，不是交易动作。后续运行02_peer_features_and_labels.ipynb生成技术/聚类特征、标签和data/events/。全部特征在T日收盘后可知。

## 1. 导入环境与读取原始表

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
FACTORS = {'510300': 'hs300', '510500': 'zz500', '159915': 'cyb',
           '588000': 'kc50', '513100': 'oversea', '518880': 'gold'}
B_FEATURES = ['alpha', *['beta_'+v for v in FACTORS.values()], 'eps', 'X',
              'kappa', 'm', 'sigma_eq', 'HL', 's', 'ols_r2']

DATA_DIR = Path('data')
OUTPUT_DIR = DATA_DIR / 'ar1'
OUTPUT_DIR.mkdir(exist_ok=True)
WINDOW = 250
PRICE_COL = 'close_hfq'

original = pd.read_parquet(DATA_DIR / 'etf_daily_full_coverage_483.parquet')
assert not original.duplicated(['date', 'code']).any()
assert original['date'].notna().all() and original['code'].notna().all()
assert (np.isfinite(original[PRICE_COL]) & original[PRICE_COL].gt(0)).all()
original = original.loc[original['date'].le('2025-12-31')].sort_values(['date', 'code']).reset_index(drop=True)
original.head()

,date,code,open,high,low,close_raw,volume,amount,turnover,pct_chg_source,...,price_adjustment_source,hfq_factor,ret_hfq_1d,hfq_return_span_sessions,core_complete,akshare_complete,four_fields_complete,is_imputed,imputed_from_date,imputation_method
0,2022-01-04,159001,100.000,100.001,99.999,100.001,4.901746e+06,4.901716e+08,NaN,0.000000,...,unverified_existing_close_hfq,NaN,NaN,NaN,True,True,False,False,NaT,none
1,2022-01-04,159003,99.979,99.979,99.960,99.969,2.664090e+05,2.663229e+07,NaN,-0.010000,...,unverified_existing_close_hfq,NaN,NaN,NaN,True,True,False,False,NaT,none
2,2022-01-04,159005,99.999,100.000,99.999,100.000,1.849500e+04,1.849484e+06,NaN,0.000000,...,unverified_existing_close_hfq,NaN,NaN,NaN,True,True,False,False,NaT,none
3,2022-01-04,159601,1.004,1.006,0.984,0.995,2.403632e+09,2.382827e+09,NaN,-0.700000,...,unverified_existing_close_hfq,NaN,NaN,NaN,True,True,False,False,NaT,none
4,2022-01-04,159602,0.999,1.005,0.982,0.995,5.397790e+08,5.340471e+08,NaN,-0.300601,...,unverified_existing_close_hfq,NaN,NaN,NaN,True,True,False,False,NaT,none


## 2. 对齐交易日并重算对数收益

$r_t=\log(P_t/P_{t-1})$。不读取原有收益列，不使用未来数据回填。

513100 的 2022-01-13 保留原表的复制价格，因此对齐收益为 0；停牌与人工填充标记不变。此处是交易所日历上的价格变化，不将停牌日视为可交易日。

In [2]:
prices = original.pivot(index='date', columns='code', values=PRICE_COL).sort_index()
assert prices.notna().all().all(), '存在缺失交易日，需检查；不自动填充'
returns = np.log(prices / prices.shift(1))
returns.index.name = 'date'
returns.columns.name = 'code'
returns.tail()

code,159001,159003,159005,159601,159602,159605,159606,159607,159608,159703,...,588150,588180,588280,588300,588310,588330,588360,588380,588390,588400
date,,,,,,,,,,,,,,,,,,,,,
2025-12-25,0.00000,0.00000,-0.00001,0.000934,0.001944,0.000000,0.005159,0.000923,-0.004755,-0.004819,...,-0.002103,-0.002286,-0.001024,-0.002079,-0.001972,-0.003170,-0.000987,-0.002094,0.000000,-0.002121
2025-12-26,0.00001,-0.00001,0.00000,0.003728,0.002908,0.003663,0.005133,0.004602,0.028197,0.010811,...,0.000000,-0.002291,-0.001025,-0.004171,0.000000,-0.002119,-0.001976,-0.004202,-0.002969,-0.001062
2025-12-29,-0.00001,0.00001,0.00000,-0.003728,-0.003880,-0.009183,-0.001708,-0.007373,-0.027245,-0.009604,...,0.000000,0.001146,0.000000,-0.001045,-0.002966,-0.001061,-0.001980,-0.003163,-0.002978,-0.004260
2025-12-30,0.00000,0.00001,0.00001,0.005587,0.003880,0.009183,0.000854,0.008291,0.011364,0.004813,...,0.009429,0.010257,0.010204,0.011441,0.008871,0.009509,0.008880,0.013634,0.009891,0.012726
2025-12-31,-0.00001,0.00000,0.00001,-0.006521,-0.004852,-0.011954,-0.001709,-0.014788,0.004697,-0.003608,...,-0.010482,-0.011403,-0.011230,-0.020899,-0.019822,-0.019109,-0.018840,-0.019990,-0.019881,-0.021300


## 3. 滚动 OLS → 局部累积残差 → AR(1)

每个 T 使用 `[T-250, T-1]` 的 250 个日收益（需要 251 个历史价格）拟合含截距的六因子 OLS：

$r_{i,t}=\alpha_{i,T}+\sum_j\beta_{ij,T}r_{j,t}+\epsilon_{i,t}^{(T)}$。

用同一次 OLS 系数重算这 250 日残差，窗口内累加得到 250 个历史 $X$ 点；以相邻的 **249 对**拟合 $X_{u+1}=a+bX_u+\eta_{u+1}$。不额外插入 $X=0$，也不使用 T 日数据估计 AR 参数。

创新标准差使用 $\sqrt{SSE/(249-2)}$。仅在 $0<b<1$ 且非退化时计算：

$\kappa=-\log b,\quad m=a/(1-b),\quad \sigma_{eq}=\sigma_\eta/\sqrt{1-b^2},\quad HL=\log(2)/\kappa$。

当日 $\epsilon_T=r_T-\alpha_T-\beta_T^T f_T$；$X_T=\sum_{u=T-250}^{T-1}\epsilon_u^{(T)}+\epsilon_T$；$s_T=(X_T-m)/\sigma_{eq}$。

含截距时历史残差和约为 0，因此局部 $X_T$ 通常接近当日残差，不能解释为全局累积错价。六只基准也保留行，但自解释残差退化时信号缺失，不以浮点噪声产生信号。

下方一个滚动函数完成逐日计算，内部依次标出历史窗口、OLS、累积残差、AR(1) 和信号五步。保留函数是为了复用同一计算过程做检查；不再依赖项目内的 Python 模块。

In [3]:
def rolling_features(returns, window=250):
    """输入完整日历的日对数收益宽表；不删除缺失日，不使用 T 日拟合参数。

    历史窗口有 window 个 X 点，AR 使用 window-1 对，不人为加入 X=0。
    创新方差为 SSE/(window-3)，扣除 AR 截距和斜率两个自由度。
    """
    assert window > 7
    assert returns.index.is_unique and returns.index.is_monotonic_increasing
    assert returns.columns.is_unique and set(FACTORS).issubset(returns.columns)
    values = returns.to_numpy(dtype=float)
    factors = returns[list(FACTORS)].to_numpy(dtype=float)
    codes = returns.columns.to_numpy()
    rows = []
    numeric = list(dict.fromkeys(B_FEATURES + ['ar1_a', 'ar1_b', 'ar1_innovation_sd']))
    for t, date in enumerate(returns.index):
        row = pd.DataFrame(np.nan, index=range(len(codes)), columns=numeric)
        row['date'], row['code'] = date, codes
        row['fit_start'], row['fit_end'] = pd.NaT, pd.NaT
        row['ols_valid'], row['ar1_valid'] = False, False
        row['signal_status'] = 'insufficient_history'
        if t < window:
            rows.append(row)
            continue
        # ① 取 T 之前的历史窗口，不包含 T 日
        row['fit_start'] = returns.index[t-window]
        row['fit_end'] = returns.index[t-1]
        f = factors[t-window:t]
        y = values[t-window:t]
        row['signal_status'] = 'missing_history'
        if not np.isfinite(f).all():
            rows.append(row)
            continue
        valid = np.isfinite(y).all(axis=0)
        ids = np.flatnonzero(valid)
        if not len(ids):
            rows.append(row)
            continue
        # ② 含截距的六因子 OLS：同一天批量计算所有有效 ETF
        design = np.column_stack([np.ones(window), f])
        coef, _, rank, _ = np.linalg.lstsq(design, y[:, valid], rcond=None)
        if rank < design.shape[1]:
            row.loc[ids, 'signal_status'] = 'singular_factor_window'
            rows.append(row)
            continue
        residual = y[:, valid] - design @ coef
        row.loc[ids, ['alpha', *['beta_'+v for v in FACTORS.values()]]] = coef.T
        row.loc[ids, 'ols_valid'] = True
        sse = (residual**2).sum(axis=0)
        sst = ((y[:, valid] - y[:, valid].mean(axis=0))**2).sum(axis=0)
        r2 = np.divide(sse, sst, out=np.full(len(ids), np.nan), where=sst>0)
        row.loc[ids, 'ols_r2'] = 1-r2
        # ③ 计算当日残差和窗口内局部累积残差 X
        eps = values[t, valid] - np.r_[1., factors[t]] @ coef
        local_x = residual.cumsum(axis=0)
        current_x = local_x[-1] + eps
        row.loc[ids, 'eps'] = eps
        row.loc[ids, 'X'] = current_x
        # ④ 用历史 X 的相邻点估计 AR(1)，不包含当日 X
        lag, lead = local_x[:-1], local_x[1:]
        lag_centered = lag - lag.mean(axis=0)
        denominator = (lag_centered**2).sum(axis=0)
        # 自基准回归的残差只有浮点误差，不允许变成虚假的 AR 信号。
        degenerate = (np.sqrt(sse/window) <= 1e-12) | (denominator <= 1e-24)
        b = np.divide((lag_centered*(lead-lead.mean(axis=0))).sum(axis=0),
                      denominator, out=np.full(len(ids), np.nan), where=~degenerate)
        a = lead.mean(axis=0) - b*lag.mean(axis=0)
        innovation = lead - a - b*lag
        sd = np.sqrt((innovation**2).sum(axis=0)/(window-3))
        row.loc[ids, ['ar1_a', 'ar1_b', 'ar1_innovation_sd']] = np.column_stack([a,b,sd])
        # ⑤ 仅对非退化且 0 < b < 1 的过程计算动力学参数和 s
        stationary = (~degenerate) & np.isfinite(a+b+sd) & (b>0) & (b<1) & (sd>1e-12)
        row.loc[ids, 'signal_status'] = 'ar1_not_mean_reverting'
        row.loc[ids[degenerate], 'signal_status'] = 'degenerate_residual'
        good = ids[stationary]
        bg, ag, sdg = b[stationary], a[stationary], sd[stationary]
        kappa = -np.log(bg)
        m = ag/(1-bg)
        sigma = sdg/np.sqrt(1-bg**2)
        score = (current_x[stationary]-m)/sigma
        row.loc[good, ['kappa','m','sigma_eq','HL','s']] = np.column_stack(
            [kappa,m,sigma,np.log(2)/kappa,score])
        row.loc[good, 'ar1_valid'] = True
        row.loc[good, 'signal_status'] = np.where(np.isfinite(score),'valid','missing_current_return')
        rows.append(row)
    return pd.concat(rows, ignore_index=True)


In [4]:
features = rolling_features(returns, window=WINDOW)
features['is_benchmark'] = features['code'].isin(FACTORS)
features['signal_valid'] = features['ar1_valid'] & np.isfinite(features['s'])
return_long = returns.stack(future_stack=True).rename('log_ret_1d').reset_index()
features = features.merge(return_long, on=['date', 'code'], validate='one_to_one')
features['price_input_policy'] = 'existing_close_hfq_as_requested'
assert len(features) == len(original)
assert not features.duplicated(['date', 'code']).any()
assert features.loc[features.ols_valid, 'fit_end'].lt(features.loc[features.ols_valid, 'date']).all()
features[['date', 'code', 'alpha', 'eps', 'X', 'ar1_a', 'ar1_b', 'kappa', 'HL', 's', 'signal_status']].tail(10)

,date,code,alpha,eps,X,ar1_a,ar1_b,kappa,HL,s,signal_status
468017,2025-12-31,588150,0.000067,0.000705,0.000705,-0.000303,0.850906,0.161453,4.293172,0.713335,valid
468018,2025-12-31,588180,0.000067,-0.000611,-0.000611,-0.000411,0.873854,0.134842,5.140442,0.652312,valid
468019,2025-12-31,588280,0.000039,0.000021,0.000021,-0.000131,0.750462,0.287067,2.414587,0.208812,valid
468020,2025-12-31,588300,0.000392,-0.002675,-0.002675,-0.000643,0.981153,0.019026,36.430753,1.618830,valid
468021,2025-12-31,588310,0.000406,-0.003024,-0.003024,-0.000848,0.974276,0.026060,26.597798,1.381360,valid
468022,2025-12-31,588330,0.000379,-0.000493,-0.000493,-0.000730,0.977416,0.022843,30.344534,1.792011,valid
468023,2025-12-31,588360,0.000403,-0.000165,-0.000165,-0.000841,0.974172,0.026168,26.488637,1.863866,valid
468024,2025-12-31,588380,0.000367,-0.001027,-0.001027,-0.000685,0.979766,0.020442,33.908082,1.791130,valid
468025,2025-12-31,588390,0.000366,-0.001758,-0.001758,-0.000733,0.978200,0.022041,31.447444,1.585936,valid
468026,2025-12-31,588400,0.000369,-0.002721,-0.002721,-0.000699,0.980613,0.019577,35.405814,1.658056,valid


## 4. 信号先保存 CSV，再读取并合并原始字段

`signals.csv` 包含完整历史及预热期，供复现和后续技术指标/聚类使用。每行使用 `date/code` 唯一关联；原始 64 列不被覆盖，新增列为本次计算结果。无效 AR 行仍然保留。

In [5]:
signal_path = OUTPUT_DIR / 'signals.csv'
features.to_csv(signal_path, index=False, date_format='%Y-%m-%d')
signals = pd.read_csv(signal_path, dtype={'code': 'string'},
                      parse_dates=['date', 'fit_start', 'fit_end'], float_precision='round_trip')
assert not signals.duplicated(['date', 'code']).any()
merged = original.merge(signals, on=['date', 'code'], how='left', validate='one_to_one')
assert len(merged) == len(original)
pd.testing.assert_frame_equal(merged[original.columns], original)
merged.head()

,date,code,open,high,low,close_raw,volume,amount,turnover,pct_chg_source,...,ar1_innovation_sd,fit_start,fit_end,ols_valid,ar1_valid,signal_status,is_benchmark,signal_valid,log_ret_1d,price_input_policy
0,2022-01-04,159001,100.000,100.001,99.999,100.001,4.901746e+06,4.901716e+08,NaN,0.000000,...,NaN,NaT,NaT,False,False,insufficient_history,False,False,NaN,existing_close_hfq_as_requested
1,2022-01-04,159003,99.979,99.979,99.960,99.969,2.664090e+05,2.663229e+07,NaN,-0.010000,...,NaN,NaT,NaT,False,False,insufficient_history,False,False,NaN,existing_close_hfq_as_requested
2,2022-01-04,159005,99.999,100.000,99.999,100.000,1.849500e+04,1.849484e+06,NaN,0.000000,...,NaN,NaT,NaT,False,False,insufficient_history,False,False,NaN,existing_close_hfq_as_requested
3,2022-01-04,159601,1.004,1.006,0.984,0.995,2.403632e+09,2.382827e+09,NaN,-0.700000,...,NaN,NaT,NaT,False,False,insufficient_history,False,False,NaN,existing_close_hfq_as_requested
4,2022-01-04,159602,0.999,1.005,0.982,0.995,5.397790e+08,5.340471e+08,NaN,-0.300601,...,NaN,NaT,NaT,False,False,insufficient_history,False,False,NaN,existing_close_hfq_as_requested


## 5. 导出两个日频来源阶段

- 训练：2023-02-02 至 2024-12-31；
- 验证：2025 年；
本轮不输出2026数据。

先在完整时间轴计算特征，再切分，不在阶段边界重置滚动历史。2022-01-04 至 2023-02-01 为预热；仅导出2023–2024和2025两个来源阶段。各表包含所有 483 只 ETF，`is_benchmark` 区分 6 只基准和 477 只候选。

这些是**每日特征表**，尚无监督学习标签 `Y`。后续生成 20 日价格标签时，必须按标签终点剔除跨阶段窗口，不能直接把末尾未知标签当失败。

In [6]:
# 日期边界固定；先计算完整历史，再切成两个来源阶段。
ranges = {'train': ('2023-02-02', '2024-12-31'),
          'validation': ('2025-01-01', '2025-12-31')}
tables = {name: merged.loc[merged.date.between(start, end)].assign(split=name)
          for name, (start, end) in ranges.items()}
summary_rows = []
for split, table in tables.items():
    table.to_parquet(OUTPUT_DIR / f'{split}.parquet', index=False)
    table.to_csv(OUTPUT_DIR / f'{split}.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    restored = pd.read_parquet(OUTPUT_DIR / f'{split}.parquet')
    pd.testing.assert_frame_equal(restored, table.reset_index(drop=True))
    assert not table.duplicated(['date', 'code']).any()
    summary_rows.append({'split': split, 'start': table.date.min(), 'end': table.date.max(),
                         'rows': len(table), 'dates': table.date.nunique(),
                         'etfs': table.code.nunique(), 'columns': table.shape[1],
                         'valid_signals': int(table.signal_valid.sum())})
summary = pd.DataFrame(summary_rows)
summary

,split,start,end,rows,dates,etfs,columns,valid_signals
0,train,2023-02-02,2024-12-31,225561,467,483,92,222755
1,validation,2025-01-02,2025-12-31,117369,243,483,92,115905


## 6. 质量检查与特征清单

In [7]:
assert tables['train'].date.max() < tables['validation'].date.min()
assert not merged[B_FEATURES].isin([np.inf, -np.inf]).any().any()
assert not merged.loc[merged.is_benchmark, 'signal_valid'].any()
assert merged.loc[merged.signal_valid, 'ar1_b'].between(0, 1, inclusive='neither').all()
assert merged.loc[merged.signal_valid, ['sigma_eq', 'HL']].gt(0).all().all()
metadata = {
    'price_column': PRICE_COL, 'price_policy': 'existing_close_hfq_as_requested_no_new_verification',
    'window': WINDOW, 'ar1_pairs': WINDOW - 1, 'innovation_variance_ddof': 2,
    'factor_codes': FACTORS, 'B_features': B_FEATURES,
    'signal': 's; continuous score, not position or trade action',
    'first_ols_date': str(features.loc[features.ols_valid, 'date'].min().date()),
    'source_rows': len(original), 'source_columns': len(original.columns),
    'tables': json.loads(summary.to_json(orient='records', date_format='iso')),
    'labels_generated': False,
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
status_summary = merged.assign(split=np.select(
    [merged.date.lt('2023-02-02'), merged.date.lt('2025-01-01'), merged.date.lt('2026-01-01')],
    ['warmup', 'train', 'validation'], default='unused')).groupby(['split', 'signal_status']).size().rename('rows').reset_index()
status_summary

,split,signal_status,rows
0,train,ar1_not_mean_reverting,4
1,train,degenerate_residual,2802
2,train,valid,222755
3,validation,ar1_not_mean_reverting,6
4,validation,degenerate_residual,1458
5,validation,valid,115905
6,warmup,degenerate_residual,48
7,warmup,insufficient_history,120750
8,warmup,missing_history,483
9,warmup,valid,3816


## 7. 读取输出表

B 类候选特征列表为 `B_FEATURES`，`s` 已包含在其中；可将它们与后续 A/C 特征关联。`code/date/split`、质量标记与诊断列不默认进入预测变量；不要把原表全部数值列直接喂给模型。

In [8]:
train = tables['train']
validation = tables['validation']
train[['date', 'code', *B_FEATURES, 'signal_status']].head()

,date,code,alpha,beta_hs300,beta_zz500,beta_cyb,beta_kc50,beta_oversea,beta_gold,eps,X,kappa,m,sigma_eq,HL,s,ols_r2,signal_status
125097,2023-02-02,159001,-2.661125e-08,0.000067,-0.000176,-8.615033e-07,0.000077,-0.000004,-0.000046,6.291710e-07,6.291710e-07,0.686599,0.000005,0.000010,1.009538,-0.455711,0.041152,valid
125098,2023-02-02,159003,-2.820766e-07,0.000078,0.000039,-2.787519e-04,0.000006,-0.000044,0.000219,2.790782e-05,2.790782e-05,0.399690,-0.000031,0.000045,1.734212,1.294306,0.014078,valid
125099,2023-02-02,159005,1.219002e-06,-0.001489,-0.000307,-1.349917e-03,0.002064,0.000076,-0.002543,1.078292e-05,1.078292e-05,1.039337,-0.000084,0.000245,0.666913,0.386868,0.013901,valid
125100,2023-02-02,159601,1.414160e-04,0.973402,-0.082309,1.993040e-01,-0.118397,0.029648,0.049547,5.729376e-04,5.729376e-04,0.021996,0.006684,0.014470,31.512653,-0.422300,0.950683,valid
125101,2023-02-02,159602,1.504665e-04,0.994273,-0.090783,1.617660e-01,-0.098523,0.038054,0.035250,3.256805e-04,3.256805e-04,0.022150,0.007670,0.015231,31.293360,-0.482220,0.944324,valid


## 附录：关键计算检查

使用固定随机种子的合成数据，独立对照 statsmodels；检查未来数据不改变过去结果、T 日收益不参与参数拟合、基准残差退化与缺失窗口。合成数据只用于检查，不参与实际输出。检查集中在一个局部函数中，避免覆盖上面的真实数据变量。

In [9]:
def check_calculation():
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm
    rng = np.random.default_rng(32)
    n=280; dates=pd.bdate_range('2020-01-01', periods=n)
    f=rng.normal(0,.01,(n,6))
    z=np.zeros(n)
    for i in range(1,n): z[i]=.85*z[i-1]+rng.normal(0,.005)
    y=f@np.arange(1,7)/10+np.diff(z,prepend=0)+.0001
    returns=pd.DataFrame(np.column_stack([f,y]),index=dates,columns=list(FACTORS)+['999999'])
    out=rolling_features(returns)
    t=270
    row=out[(out.date==dates[t]) & (out.code=='999999')].iloc[0]
    ols=sm.OLS(y[t-250:t],sm.add_constant(f[t-250:t])).fit()
    x=ols.resid.cumsum()
    ar=sm.OLS(x[1:],sm.add_constant(x[:-1])).fit()
    np.testing.assert_allclose(row[['alpha',*['beta_'+v for v in FACTORS.values()]]].astype(float),ols.params,atol=1e-12)
    np.testing.assert_allclose([row.ar1_a,row.ar1_b,row.ar1_innovation_sd],[*ar.params,np.sqrt(ar.mse_resid)],rtol=1e-9)
    xt=x[-1]+y[t]-np.r_[1,f[t]]@ols.params
    s=(xt-ar.params[0]/(1-ar.params[1]))/(np.sqrt(ar.mse_resid)/np.sqrt(1-ar.params[1]**2))
    np.testing.assert_allclose(row.s,s,rtol=1e-9)
    changed=returns.copy();changed.iloc[t+1:]*=4
    pd.testing.assert_frame_equal(out[out.date<=dates[t]].reset_index(drop=True),rolling_features(changed).loc[lambda frame: frame.date <= dates[t]].reset_index(drop=True))
    assert not out[out.code.isin(FACTORS)].ar1_valid.any()
    missing=returns.copy();missing.loc[dates[100],'999999']=np.nan
    assert not rolling_features(missing).query("code == '999999'").ols_valid.any()
    print('PASS: statsmodels reference, future perturbation, benchmark degeneracy, missing windows')

    # T 日改变只影响当前残差/信号，不影响历史 OLS 和 AR 参数。
    changed = returns.copy()
    changed.loc[dates[t], '999999'] += .02
    current = rolling_features(changed)
    current_row = current[(current.date == dates[t]) & (current.code == '999999')].iloc[0]
    for col in ['alpha', 'ar1_a', 'ar1_b', 'ar1_innovation_sd', 'm', 'HL']:
        np.testing.assert_allclose(current_row[col], row[col], rtol=1e-10)
    np.testing.assert_allclose(current_row.eps - row.eps, .02, atol=1e-12)


check_calculation()

PASS: statsmodels reference, future perturbation, benchmark degeneracy, missing windows


## 最终数据表与 Summary

`final_ar1_data` 保留两个正式来源阶段的全部导出列，不含历史预热行及2026年数据。下表展示前10行的主要字段；完整表可直接查看该变量。Summary 按来源阶段汇总样本和有效性，再逐项列出残差特征的分布及缺失比例。无效AR1结果保留在交付表中，由后续步骤按研究所需特征筛选。

In [ ]:
# 与上面导出的 train / validation 内容一致，只合并用于展示。
final_ar1_data = pd.concat(tables.values(), ignore_index=True)
pd.set_option('display.max_columns', 30)
final_ar1_data[['date', 'code', 'split', 'close_hfq', *B_FEATURES,
                'signal_valid', 'signal_status']].head(10)

,date,code,split,close_hfq,alpha,beta_hs300,beta_zz500,beta_cyb,beta_kc50,beta_oversea,beta_gold,eps,X,kappa,m,sigma_eq,HL,s,ols_r2,signal_valid,signal_status
0,2023-02-02,159001,train,100.000,-2.661125e-08,0.000067,-0.000176,-8.615033e-07,0.000077,-0.000004,-0.000046,6.291710e-07,6.291710e-07,0.686599,0.000005,0.000010,1.009538,-0.455711,0.041152,True,valid
1,2023-02-02,159003,train,100.002,-2.820766e-07,0.000078,0.000039,-2.787519e-04,0.000006,-0.000044,0.000219,2.790782e-05,2.790782e-05,0.399690,-0.000031,0.000045,1.734212,1.294306,0.014078,True,valid
2,2023-02-02,159005,train,100.000,1.219002e-06,-0.001489,-0.000307,-1.349917e-03,0.002064,0.000076,-0.002543,1.078292e-05,1.078292e-05,1.039337,-0.000084,0.000245,0.666913,0.386868,0.013901,True,valid
3,2023-02-02,159601,train,0.878,1.414160e-04,0.973402,-0.082309,1.993040e-01,-0.118397,0.029648,0.049547,5.729376e-04,5.729376e-04,0.021996,0.006684,0.014470,31.512653,-0.422300,0.950683,True,valid
4,2023-02-02,159602,train,0.876,1.504665e-04,0.994273,-0.090783,1.617660e-01,-0.098523,0.038054,0.035250,3.256805e-04,3.256805e-04,0.022150,0.007670,0.015231,31.293360,-0.482220,0.944324,True,valid
5,2023-02-02,159605,train,0.901,8.496815e-04,2.423744,-0.308246,-2.478287e-01,-0.202871,0.424150,-0.012668,3.131991e-03,3.131991e-03,0.072546,-0.021696,0.055123,9.554574,0.450415,0.610246,True,valid
6,2023-02-02,159606,train,0.904,2.474906e-04,0.547793,0.290496,1.996290e-02,0.070157,-0.027278,0.095677,1.240932e-03,1.240932e-03,0.056364,0.023298,0.020527,12.297657,-1.074515,0.751350,True,valid
7,2023-02-02,159607,train,0.906,8.473355e-04,2.425192,-0.332175,-2.092958e-01,-0.219018,0.413738,-0.053382,6.144614e-03,6.144614e-03,0.073059,-0.019094,0.055893,9.487539,0.451551,0.602605,True,valid
8,2023-02-02,159608,train,0.867,6.877389e-04,0.340924,0.300853,3.422873e-01,0.243879,-0.048150,0.047518,-5.939979e-03,-5.939979e-03,0.027817,0.092798,0.065630,24.918529,-1.504454,0.545168,True,valid
9,2023-02-02,159703,train,0.964,1.210324e-04,0.123914,0.047219,6.854799e-01,0.231174,-0.028017,-0.046592,6.755139e-03,6.755139e-03,0.013549,0.062199,0.047346,51.157558,-1.171039,0.828465,True,valid


In [ ]:
# Summary：正式数据覆盖范围，以及不同来源阶段的有效残差信号。
final_ar1_summary = final_ar1_data.groupby('split', sort=False).agg(
    rows=('code', 'size'), etfs=('code', 'nunique'), dates=('date', 'nunique'),
    date_start=('date', 'min'), date_end=('date', 'max'),
    valid_signals=('signal_valid', 'sum'), valid_signal_rate=('signal_valid', 'mean'))
final_ar1_summary['columns'] = final_ar1_data.shape[1]
final_ar1_summary

,rows,etfs,dates,date_start,date_end,valid_signals,valid_signal_rate,columns
split,,,,,,,,
train,225561,483,467,2023-02-02,2024-12-31,222755,0.987560,92
validation,117369,483,243,2025-01-02,2025-12-31,115905,0.987527,92


In [ ]:
# 缺失值不填成零；统计量按每列实际非缺失数据计算。
final_ar1_feature_summary = final_ar1_data[B_FEATURES].describe().T
final_ar1_feature_summary['missing_count'] = final_ar1_data[B_FEATURES].isna().sum()
final_ar1_feature_summary['missing_rate'] = final_ar1_data[B_FEATURES].isna().mean()
final_ar1_feature_summary

,count,mean,std,min,25%,50%,75%,max,missing_count,missing_rate
alpha,342930.0,0.000042,0.000661,-0.011085,-1.781067e-04,2.326995e-05,0.000247,0.009286,0,0.000000
beta_hs300,342930.0,0.375564,0.587511,-1.462382,-3.644320e-02,2.222202e-01,0.856803,5.456824,0,0.000000
beta_zz500,342930.0,0.258784,0.386042,-2.619489,-3.070137e-16,1.453773e-01,0.497460,3.349488,0,0.000000
beta_cyb,342930.0,0.146144,0.427499,-3.498242,-8.793091e-02,5.044925e-02,0.342427,1.947504,0,0.000000
beta_kc50,342930.0,0.089411,0.327046,-1.763762,-9.505701e-02,-2.261280e-03,0.134771,1.395925,0,0.000000
beta_oversea,342930.0,0.047730,0.132969,-0.621394,-1.144421e-02,1.068091e-02,0.068050,1.090267,0,0.000000
beta_gold,342930.0,0.019801,0.190544,-2.027826,-4.807948e-02,-4.827437e-16,0.040957,1.171885,0,0.000000
eps,342930.0,-0.000093,0.011701,-0.968632,-3.713552e-03,-4.252557e-05,0.003066,2.302627,0,0.000000
X,342930.0,-0.000093,0.011701,-0.968632,-3.713552e-03,-4.252557e-05,0.003066,2.302627,0,0.000000
kappa,338660.0,0.088401,0.179232,0.000049,2.660339e-02,4.188181e-02,0.068558,4.103544,4270,0.012452
